# show-o2-space
Select **Python 3, A100 GPU, High-RAM** when available. T4 memory/precision support remains experimental.
All inference cells run ordinary Python in Colab's own kernel and reuse one loaded adapter.


In [ ]:
import subprocess
print(subprocess.check_output(["nvidia-smi"], text=True))
print(subprocess.check_output(["free", "-h"], text=True))


## 1. Clone project and original model implementation
Run setup once per fresh Colab runtime. This never deletes an existing checkout.


In [ ]:
from pathlib import Path
import os
import subprocess

def run(*args):
    subprocess.run(args, check=True)

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
if not PROJECT.exists():
    run("git", "clone", "https://github.com/Nicerova7/show-o2-space.git", str(PROJECT))
if not UPSTREAM.exists():
    run("git", "clone", "https://github.com/showlab/Show-o.git", str(UPSTREAM))
revision = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"
run("git", "-C", str(UPSTREAM), "checkout", revision)
# Also expose the original packages to standalone CLI commands.
os.environ["PYTHONPATH"] = str(UPSTREAM / "show-o2") + os.pathsep + str(PROJECT)
os.chdir(PROJECT)


In [ ]:
# Install into Colab's own Python; keep its matched torch/torchvision packages.
import sys
run(
    sys.executable, "-m", "pip", "install",
    "transformers==4.47.0",
    "diffusers==0.31.0",
    "huggingface-hub==0.27.1",
    "accelerate==1.2.1",
    "timm==1.0.12",
    "einops==0.8.0",
    "torchdiffeq==0.2.5",
    "numpy>=1.26,<3",
    "Pillow>=10.4,<12",
    "sentencepiece==0.2.1",
)
print("Now select Runtime > Restart session, then continue with the next cell.")


### Restart once, then load the model
After installation, select **Runtime > Restart session** (not Disconnect and delete runtime). Continue below; do not rerun installation. The next cell restores paths and loads the adapter once. Later reruns reuse it.
This setup uses Colab's installed Python/PyTorch rather than the previous isolated environment. Compatibility and GPU inference with this environment still need validation.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from IPython.display import Image, display

# Restore paths after restarting the Colab session.
PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(UPSTREAM / "show-o2") + os.pathsep + str(PROJECT)
for path in (str(PROJECT), str(UPSTREAM / "show-o2")):
    if path not in sys.path:
        sys.path.insert(0, path)

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter
from show_o2_space.capture import Capture

print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
if "adapter" not in globals():
    adapter = ShowO2Adapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Adapter ready; sections 3 and 4 reuse this instance.")


## 2. Choose an image
Use the upstream example, or run the optional upload cell.


In [ ]:
import shutil
images = PROJECT / "inputs/images"
images.mkdir(parents=True, exist_ok=True)
source = UPSTREAM / "show-o2/docs/mmu/pexels-jane-pham-727419-1571673.jpg"
shutil.copy2(source, images / "example.jpg")
from IPython.display import display, Image
display(Image(filename=str(images / "example.jpg"), width=432))


In [ ]:
# Optional: upload a JPG or PNG from your computer.
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    destination = images / Path(name).name
    destination.write_bytes(data)
    print("Use this image_path in section 3:", str(destination.relative_to(PROJECT)))


## 3. Understanding and hidden states
Edit the image path and question below. Extract blocks 0, 8, and 16 in one prompt forward pass, without generating an answer. Rerun this cell after changing either input.


In [ ]:
image_path = "inputs/images/example.jpg"
question = "Describe this image."
layers = [0, 8, 16]
seed = 42

torch.manual_seed(seed)
embeddings, mask, alignment = adapter.prepare_understanding(image_path, question)
with Capture(adapter.layers, layers) as captured:
    result = adapter.forward_understanding(embeddings, mask)
del result  # Only the selected CPU states are needed.

understanding_output = Path("outputs/understanding")
understanding_output.mkdir(parents=True, exist_ok=True)
# A new extraction has no decoded answer yet.
(understanding_output / "answer.txt").unlink(missing_ok=True)
understanding_metadata = dict(
    model=MODEL_ID, resolution=432, image=str(Path(image_path).resolve()),
    question=question, layers=layers, seed=seed, dtype=str(adapter.dtype),
    capture="single prefill forward; decoder-block outputs before final norm",
    alignment=alignment,
)
torch.save(
    {"hidden_states": captured.states, "metadata": understanding_metadata},
    understanding_output / "hidden_states.pt",
)
(understanding_output / "metadata.json").write_text(
    json.dumps(understanding_metadata, indent=2), encoding="utf-8",
)
for layer, hidden in captured.states.items():
    print(f"Layer {layer}: {tuple(hidden.shape)}")
print("Saved prompt hidden states to", understanding_output)


### Optional: generate an answer
Run after extraction if you want a textual answer. This reuses the prepared image/question and loaded adapter, with no capture hooks. The original Show-o2 sampler is unchanged; decoding can still be slow.


In [ ]:
answer = adapter.answer(embeddings, mask, max_new_tokens=128)
(understanding_output / "answer.txt").write_text(answer + "\n", encoding="utf-8")
print(answer)


## 4. Text-to-image generation
Reuse the same loaded adapter. This saves the image and the selected decoder states from the first velocity evaluation. These are noisy-image states at the start of sampling, not final-image representations.


In [ ]:
prompt = "A photograph of a golden retriever sitting on green grass."
steps = 50
guidance = 5.0
seed = 42
layers = [0, 8, 16]

with Capture(adapter.layers, layers) as captured:
    image = adapter.generate(prompt, steps=steps, guidance=guidance, seed=seed)

generation_output = Path("outputs/generation")
generation_output.mkdir(parents=True, exist_ok=True)
image.save(generation_output / "image.png")
generation_metadata = dict(
    model=MODEL_ID, resolution=432, prompt=prompt, steps=steps,
    guidance=guidance, seed=seed, layers=layers, dtype=str(adapter.dtype),
    capture="first velocity evaluation; decoder-block outputs before final norm",
    batch_order=["conditional", "unconditional"] if guidance > 0 else ["conditional"],
    prompt_token_limit=290,
)
(generation_output / "metadata.json").write_text(
    json.dumps(generation_metadata, indent=2), encoding="utf-8",
)
torch.save(
    {"hidden_states": captured.states, "metadata": generation_metadata},
    generation_output / "hidden_states.pt",
)
display(image)
print("Saved image and hidden states to", generation_output)


## 5. Download outputs
Colab storage is temporary. Save the archive before disconnecting.


In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/show-o2-space-outputs", "zip", PROJECT / "outputs")
files.download(archive)
